# The Agent Loop, Explained

> **Companion notebook** to [The AI agent loop: the architecture powering autonomous AI](BLOG_URL) — Oracle Developer Blog

---

Imagine you ask an AI assistant: *"Find the three cheapest flights to Tokyo next month, check if my loyalty points cover any of them, and book the best option."*

A standard chatbot fails this task immediately. It can answer the first part, but then it stops — waiting for your next message, having already forgotten the context. It is a bit like a shop assistant who answers your question and then immediately forgets you exist. Useful for one-off queries; useless for anything that requires follow-through.

The reason is architectural. A single-pass chatbot has three hard limitations:

1. **It cannot act.** It generates text; it cannot call APIs, run code, or change state in external systems.
2. **It cannot adapt.** There is no feedback loop. If the first approach fails, it cannot try a different one.
3. **It cannot decompose.** Multi-step tasks require breaking a problem into sub-problems, solving each in order, and accumulating results.

The **agent loop** is the architectural fix for all three. It is deceptively simple:

```python
while not done:
    response = call_llm(messages)
    if response has tool_calls:
        results = execute_tools(response.tool_calls)
        messages.append(results)
    else:
        done = True
        return response
```

An LLM calling tools inside a `while` loop, iterating until the task is complete. Every major AI agent — from OpenAI's Agents SDK to Google's Deep Research to Anthropic's Claude Code — is built on this same pattern.

This notebook makes that loop **visible and concrete**. Using three simple tools — a calculator, a unit converter, and a timezone converter — you will watch the loop execute step by step. Oracle AI Database 26ai stores every iteration so you can query your agent's full execution history with SQL.

## What You Will Learn

1. What the five stages of the agent loop are and how they map to real code
2. How LangChain's `create_agent` implements the `while` loop pattern as a compiled graph
3. How to define tools that an agent can call and observe
4. How to make the loop visible — streaming every step of reasoning and tool use live
5. How to store and query agent execution traces in Oracle AI Database 26ai

## Prerequisites

| Requirement | Details |
|-------------|---------|
| Python | 3.10 or higher |
| Podman or Docker | To run Oracle AI Database 26ai locally (free container image) |
| LLM API access | **Option 1:** Free Google AI Studio key from [aistudio.google.com/apikey](https://aistudio.google.com/apikey) — or — **Option 2:** OCI account with Generative AI enabled |
| Python packages | Installed in the next cell |

> **Minimal footprint:** This notebook uses five packages and no vector store. It is designed to be the simplest possible introduction to the agent loop before you move on to RAG, memory, and multi-agent notebooks.

## What Is the Agent Loop?

### Single-turn chatbot vs. agent loop

A traditional LLM interaction is **stateless and single-turn**:

```
User  ──► LLM  ──► Response
          (done)
```

An agent loop is **iterative and stateful**:

```
                  ┌─────────────────────────────────────────────┐
                  │               AGENT LOOP                    │
                  │                                             │
User Task ──────► │  PERCEIVE ──► REASON ──► ACT ──► OBSERVE   │
                  │     ▲                              │        │
                  │     └──────────────────────────────┘        │
                  │           (repeat until done)               │
                  └─────────────────────────────────────────────┘
                                    │
                               Final Answer
```

### The five stages

| Stage | What happens | In this notebook |
|-------|-------------|------------------|
| **Perceive** | The agent receives input from its environment | User's question + tool results from previous iterations |
| **Reason** | The LLM decides what to do next | The LLM reads the conversation and chooses a tool |
| **Plan** | The agent optionally decomposes the task | Done implicitly by the LLM in the system prompt |
| **Act** | The agent executes a tool | `calculate`, `convert_units`, or `timezone_convert` runs |
| **Observe** | The agent reads the result and loops | Tool output goes back into the message history |

### Why this matters

The loop is what enables an agent to:
- Break a complex problem into steps it can solve one at a time
- Recover from errors by trying a different approach
- Accumulate information across multiple tool calls
- Stop when it has a complete answer — not before

As Anthropic's engineering team wrote: *"Agents are typically just LLMs using tools based on environmental feedback in a loop."* The sophistication is in the tools and the quality of reasoning — not in the loop itself.

### The research behind the pattern

In 2022, researchers at Google and Princeton formalised this loop in a paper called **ReAct** (Reasoning + Acting). By interleaving chain-of-thought reasoning with tool use — rather than separating them — ReAct-style agents improved task completion by 34% on ALFWorld benchmarks compared to reasoning-only approaches.

Since then, every major AI platform has converged on the same core structure: OpenAI's Agents SDK, Anthropic's Claude, Google's Gemini agents, Microsoft's Semantic Kernel, and LangChain all implement the same PERCEIVE → REASON → ACT → OBSERVE cycle under different names. The loop is settled science. What varies is the scaffolding around it — tools, memory, observability, and multi-agent coordination.

In [ ]:
# Install required packages
# ─────────────────────────────────────────────────────────────────────────────
# After this cell finishes, Jupyter may restart the kernel automatically.
# If it does, you MUST re-run ALL cells from the top before continuing.
# The restart is normal — it ensures the newly installed packages are loaded.

%pip install -q \
    oracledb \
    "langchain>=1.0" \
    "langchain-community>=0.3" \
    "langchain-google-genai" \
    "langchain-oci" \
    python-dotenv

print("Packages installed.")
print("If the kernel restarted, re-run all cells from alf-05-config before continuing.")

## Configuration

Choose one LLM provider and configure the Oracle Database connection. Create a `.env` file in the same directory as this notebook.

**Option 1 — Google AI Studio** (free, no OCI account needed):

```
GOOGLE_API_KEY=your-api-key-here
LLM_MODEL=gemini-2.0-flash
```

**Option 2 — OCI Generative AI** (enterprise, Oracle-native):

```
LLM_PROVIDER=oci
OCI_COMPARTMENT_ID=ocid1.compartment.oc1..your-compartment-id
OCI_MODEL_ID=meta.llama-4-scout-17b-16e-instruct
OCI_SERVICE_ENDPOINT=https://inference.generativeai.us-chicago-1.oci.oraclecloud.com
```

**Oracle AI Database 26ai** (required for both options):

```
ORACLE_USER=system
ORACLE_PASSWORD=YourPassword123
ORACLE_DSN=localhost:1521/FREE
```

> `LLM_PROVIDER` defaults to `google` if not set. Set it to `oci` to switch providers without changing any other code.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)  # override=True ensures .env values win over shell env vars

# ── LLM provider selection ────────────────────────────────────────────────────
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "google")   # "google" or "oci"

# ── Option 1: Google AI Studio ────────────────────────────────────────────────
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "your-api-key-here")
LLM_MODEL      = os.getenv("LLM_MODEL",      "gemini-2.0-flash")

# ── Option 2: OCI Generative AI ───────────────────────────────────────────────
OCI_COMPARTMENT_ID   = os.getenv("OCI_COMPARTMENT_ID",   "your-compartment-ocid")
OCI_MODEL_ID         = os.getenv("OCI_MODEL_ID",         "meta.llama-4-scout-17b-16e-instruct")
OCI_SERVICE_ENDPOINT = os.getenv("OCI_SERVICE_ENDPOINT", "https://inference.generativeai.us-chicago-1.oci.oraclecloud.com")

# ── Oracle AI Database 26ai ───────────────────────────────────────────────────
ORACLE_USER     = os.getenv("ORACLE_USER",     "system")
ORACLE_PASSWORD = os.getenv("ORACLE_PASSWORD", "YourPassword123")
ORACLE_DSN      = os.getenv("ORACLE_DSN",      "localhost:1521/FREE")

print(f"LLM provider : {LLM_PROVIDER}")
if LLM_PROVIDER == "oci":
    print(f"OCI model    : {OCI_MODEL_ID}")
    print(f"OCI endpoint : {OCI_SERVICE_ENDPOINT}")
    print(f"Compartment  : {'set' if OCI_COMPARTMENT_ID != 'your-compartment-ocid' else 'NOT SET — add OCI_COMPARTMENT_ID to .env'}")
else:
    print(f"LLM model    : {LLM_MODEL}")
    print(f"API key      : {'set' if GOOGLE_API_KEY != 'your-api-key-here' else 'NOT SET — add GOOGLE_API_KEY to .env'}")
print(f"Oracle DSN   : {ORACLE_DSN}")

## Oracle AI Database 26ai Setup

Oracle AI Database 26ai is Oracle's latest database release with native AI capabilities built in, including vector search, JSON Duality Views, and SQL Property Graph. In this notebook we use it to store an execution trace of the agent loop: every tool call, its inputs, and its outputs, queryable as standard SQL.

**Start the database with Podman (Oracle Linux / RHEL):**

```bash
podman run -d \
  --name oracle-26ai \
  -p 1521:1521 \
  -e ORACLE_PWD=YourPassword123 \
  container-registry.oracle.com/database/free:latest
```

**Or with Docker (macOS / Windows / Ubuntu):**

```bash
docker run -d \
  --name oracle-26ai \
  -p 1521:1521 \
  -e ORACLE_PWD=YourPassword123 \
  container-registry.oracle.com/database/free:latest
```

> **Image tag:** `:latest` always pulls the current release, which is fine for getting started. For a reproducible environment, replace `:latest` with a specific version tag (e.g. `23.7.0.0`). Available tags are listed at [container-registry.oracle.com](https://container-registry.oracle.com/ords/ocr/ba/database/free).

> **ORACLE_PWD must match ORACLE_PASSWORD.** The `ORACLE_PWD` value you pass to the container sets the password for the `system` account. The `ORACLE_PASSWORD` variable in alf-05-config must be identical — a mismatch causes `ORA-01017: invalid username/password`. If you changed one, change both.

> The database takes approximately 60 seconds to initialise on first launch. Monitor progress with `podman logs -f oracle-26ai` (or `docker logs -f oracle-26ai`). Proceed once you see `DATABASE IS READY TO USE!`

> **Privileges:** The connection uses the `system` account by default, which has `CREATE TABLE` permission. If you switch to a custom schema user, grant `CREATE TABLE` and `INSERT` to that user before running the next cell.

In [ ]:
import oracledb

# ── Connection pool ───────────────────────────────────────────────────────────
# A pool rather than a single connection means later cells can acquire and
# release connections independently without holding one open the whole time.
pool = oracledb.create_pool(
    user=ORACLE_USER,
    password=ORACLE_PASSWORD,
    dsn=ORACLE_DSN,
    min=2,
    max=10,
    increment=1,
)

# ── Schema ────────────────────────────────────────────────────────────────────
# Wrapped in a PL/SQL block so we can swallow the "table already exists" error
# (ORA-00955) without masking genuine failures such as permission errors.
CREATE_TABLE_SQL = """
BEGIN
    EXECUTE IMMEDIATE '
        CREATE TABLE agent_traces (
            run_id      VARCHAR2(36)   NOT NULL,
            iteration   NUMBER         NOT NULL,
            tool_called VARCHAR2(100),
            tool_input  CLOB,
            tool_result CLOB,
            tokens_used NUMBER,
            ts          TIMESTAMP DEFAULT SYSTIMESTAMP
        )
    ';
EXCEPTION
    WHEN OTHERS THEN
        IF SQLCODE = -955 THEN
            NULL;  -- ORA-00955: table already exists, safe to ignore
        ELSE
            RAISE; -- Surface any other error (e.g. ORA-01031 insufficient privileges)
        END IF;
END;
"""

try:
    with pool.acquire() as conn:
        with conn.cursor() as cur:
            cur.execute(CREATE_TABLE_SQL)
        conn.commit()
    print("Connected to Oracle AI Database 26ai.")
    print("agent_traces table is ready.")
except oracledb.DatabaseError as e:
    error, = e.args
    if error.code == 1031:
        print("ERROR ORA-01031: insufficient privileges.")
        print("The connected user does not have CREATE TABLE permission.")
        print("Either connect as 'system', or grant privileges to your user:")
        print("  GRANT CREATE TABLE TO <your_user>;")
    elif error.code == 1017:
        print("ERROR ORA-01017: invalid username/password.")
        print("Check that ORACLE_PASSWORD in alf-05-config matches the")
        print("ORACLE_PWD value used when starting the container.")
    else:
        raise

## Option 1: Google AI Studio Setup

Google AI Studio gives you access to Gemini models via a simple API key — no GCP project, no billing setup, no `gcloud` commands required.

**Step 1: Get an API key**

Go to [aistudio.google.com/apikey](https://aistudio.google.com/apikey) and sign in with your Google account. Click **Create API key**, choose any project (or let it create one), and copy the key.

**Step 2: Add it to your `.env` file**

```
GOOGLE_API_KEY=your-api-key-here
```

That is all. The `langchain-google-genai` package reads this key and connects to the Gemini API automatically.

> **Free tier:** Google AI Studio has a generous free quota (1,500 requests/day on Gemini 1.5 Flash, 10 requests/minute on Gemini 2.0 Flash as of early 2026). Running this notebook comfortably fits within the free limits.

## Option 2: OCI Generative AI Setup

OCI Generative AI gives you access to hosted foundation models — including Meta Llama 4, Cohere Command, and others — running inside Oracle Cloud. Authentication uses your existing OCI credentials; no separate API key is needed.

**Step 1: Find your Compartment OCID**

In the [OCI Console](https://cloud.oracle.com), go to **Identity & Security → Compartments**. Copy the OCID of the compartment where Generative AI is enabled (the root compartment works for testing).

**Step 2: Set up OCI CLI authentication**

If you have not already configured the OCI CLI, run:

```bash
pip install oci-cli
oci setup config
```

This creates `~/.oci/config` with your tenancy, user, and key details. The `langchain-oci` package reads this file automatically — no credentials go in `.env`.

**Step 3: Add variables to your `.env` file**

```
LLM_PROVIDER=oci
OCI_COMPARTMENT_ID=ocid1.compartment.oc1..your-compartment-id
OCI_MODEL_ID=meta.llama-4-scout-17b-16e-instruct
OCI_SERVICE_ENDPOINT=https://inference.generativeai.us-chicago-1.oci.oraclecloud.com
```

**Available region endpoints**

| Region | Endpoint |
|--------|----------|
| US Midwest (Chicago) | `https://inference.generativeai.us-chicago-1.oci.oraclecloud.com` |
| Germany Central (Frankfurt) | `https://inference.generativeai.eu-frankfurt-1.oci.oraclecloud.com` |
| UK South (London) | `https://inference.generativeai.uk-london-1.oci.oraclecloud.com` |
| Brazil East (São Paulo) | `https://inference.generativeai.sa-saopaulo-1.oci.oraclecloud.com` |
| Singapore | `https://inference.generativeai.ap-singapore-1.oci.oraclecloud.com` |

> **OCI Free Tier:** New OCI accounts include free Generative AI credits. See [oracle.com/cloud/free](https://www.oracle.com/cloud/free/) for current limits.

> **Model availability varies by region.** Chicago has the broadest model selection. Check the [OCI Generative AI model list](https://docs.oracle.com/en-us/iaas/Content/generative-ai/pretrained-models.htm) for your region.

In [ ]:
if LLM_PROVIDER == "oci":
    # ── OCI Generative AI ─────────────────────────────────────────────────────
    # Auth comes from ~/.oci/config — no API key needed here.
    from langchain_oci.chat_models import ChatOCIGenAI

    if OCI_COMPARTMENT_ID == "your-compartment-ocid":
        raise ValueError(
            "OCI_COMPARTMENT_ID is not set. Add it to your .env file:\n"
            "  OCI_COMPARTMENT_ID=ocid1.compartment.oc1..your-compartment-id\n"
            "Find it in the OCI Console under Identity & Security → Compartments."
        )

    llm = ChatOCIGenAI(
        model_id=OCI_MODEL_ID,
        service_endpoint=OCI_SERVICE_ENDPOINT,
        compartment_id=OCI_COMPARTMENT_ID,
        model_kwargs={"temperature": 0, "max_tokens": 2048},
    )

else:
    # ── Google AI Studio (default) ────────────────────────────────────────────
    from langchain_google_genai import ChatGoogleGenerativeAI

    if GOOGLE_API_KEY == "your-api-key-here":
        raise ValueError(
            "GOOGLE_API_KEY is not set. Add it to your .env file:\n"
            "  GOOGLE_API_KEY=your-api-key\n"
            "Get a free key at https://aistudio.google.com/apikey"
        )

    llm = ChatGoogleGenerativeAI(
        model=LLM_MODEL,
        google_api_key=GOOGLE_API_KEY,
    )

# ── Smoke test — same for both providers ─────────────────────────────────────
# Confirm the LLM is reachable before building the agent.
# Gemini can return content as a list of parts, so normalise to string.
test = llm.invoke("Reply with exactly these three words: agent loop ready")
content = test.content if isinstance(test.content, str) else " ".join(
    p.get("text", str(p)) if isinstance(p, dict) else str(p) for p in test.content
)
print(f"LLM check: {content.strip()}")

## The Agent's Tools

Tools are the mechanism by which the agent **acts** on the world. In the `ACT` stage of the loop, the LLM decides which tool to call and with what arguments. The result goes back into the message history as an `OBSERVE` event — the agent reads it and decides what to do next.

We define three tools. They are intentionally simple — pure Python, no external APIs, no authentication — so the **loop mechanics** are the focus, not the tools themselves.

### Tool 1: `calculate`

Evaluates a mathematical expression. The agent uses this for arithmetic it cannot do reliably in its head.

In [ ]:
import math
from langchain_core.tools import tool


@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression and return the numeric result.

    Supports standard arithmetic (+, -, *, /), powers (**), and common math
    functions: sqrt, floor, ceil, round, abs, min, max.

    Args:
        expression: A mathematical expression string, e.g. "5570 / 900" or "round(6.19 * 60, 1)"

    Returns:
        The numeric result as a string, or an error message.

    Note:
        Uses eval() with a restricted namespace. Suitable for demo purposes only —
        do not pass untrusted user input directly to this function in production.
    """
    safe_builtins = {
        name: getattr(math, name)
        for name in dir(math)
        if not name.startswith("__")
    }
    safe_builtins.update({
        "abs": abs, "round": round, "int": int, "float": float,
        "min": min, "max": max,
    })
    try:
        result = eval(expression, {"__builtins__": {}}, safe_builtins)
        if isinstance(result, float):
            return str(round(result, 6))
        return str(result)
    except Exception as e:
        return f"Error evaluating '{expression}': {e}"


# Verify the tool works before connecting it to the agent
print("calculate('5570 / 900') →", calculate.invoke({"expression": "5570 / 900"}))
print("calculate('round(6.188889 * 60, 2)') →", calculate.invoke({"expression": "round(6.188889 * 60, 2)"}))

### Tool 2: `convert_units`

Converts a value between units of distance, speed, or time. The agent calls this when the problem involves units that need to change — kilometres to miles, hours to minutes, km/h to knots, and so on.

In [ ]:
@tool
def convert_units(value: float, from_unit: str, to_unit: str) -> str:
    """Convert a numeric value from one unit to another.

    Supported unit groups:
      Distance : km, miles, meters, feet, nautical miles
      Speed    : km/h, mph, m/s, knots
      Time     : hours, minutes, seconds

    Units within the same group can be converted freely.
    Cross-group conversions (e.g. km to hours) will return an error.

    Args:
        value     : The numeric value to convert
        from_unit : Source unit, e.g. "km/h", "miles", "hours"
        to_unit   : Target unit, e.g. "mph", "km",   "minutes"

    Returns:
        The converted value with target unit label, or an error message.
    """
    # Conversion factors relative to base unit per group
    # (distance base = km, speed base = km/h, time base = hours)
    to_base = {
        "km": 1.0,          "miles": 1.60934,    "meters": 0.001,
        "feet": 0.0003048,  "nautical miles": 1.852,
        "km/h": 1.0,        "mph": 1.60934,      "m/s": 3.6,
        "knots": 1.852,
        "hours": 1.0,       "minutes": 1 / 60,   "seconds": 1 / 3600,
    }
    groups = {
        "distance": {"km", "miles", "meters", "feet", "nautical miles"},
        "speed":    {"km/h", "mph", "m/s", "knots"},
        "time":     {"hours", "minutes", "seconds"},
    }

    f, t = from_unit.lower().strip(), to_unit.lower().strip()

    from_grp = next((g for g, units in groups.items() if f in units), None)
    to_grp   = next((g for g, units in groups.items() if t in units), None)

    if from_grp is None:
        return f"Unknown unit '{from_unit}'. Supported: {', '.join(sorted(to_base))}"
    if to_grp is None:
        return f"Unknown unit '{to_unit}'. Supported: {', '.join(sorted(to_base))}"
    if from_grp != to_grp:
        return f"Cannot convert {from_grp} unit '{from_unit}' to {to_grp} unit '{to_unit}'."

    result = value * to_base[f] / to_base[t]
    return f"{round(result, 4)} {to_unit}"


print("convert_units(6.188889, 'hours', 'minutes') →",
      convert_units.invoke({"value": 6.188889, "from_unit": "hours", "to_unit": "minutes"}))
print("convert_units(900, 'km/h', 'mph') →",
      convert_units.invoke({"value": 900, "from_unit": "km/h", "to_unit": "mph"}))

### Tool 3: `timezone_convert`

Converts a local time from one city to another. The agent uses this when the problem spans time zones — converting a departure time to an arrival time in a different city, for example.

In [ ]:
@tool
def timezone_convert(time_str: str, from_city: str, to_city: str) -> str:
    """Convert a local time from one city's timezone to another.

    Uses standard (non-DST) UTC offsets. Supported cities include:
      UTC+0  : London, Dublin, Accra
      UTC+1  : Paris, Berlin, Amsterdam, Rome, Madrid
      UTC+2  : Cairo, Helsinki, Athens
      UTC+3  : Moscow, Riyadh, Nairobi
      UTC+4  : Dubai
      UTC+5.5: Mumbai, New Delhi, Kolkata
      UTC+8  : Singapore, Hong Kong, Beijing, Shanghai, Perth
      UTC+9  : Tokyo, Seoul
      UTC+10 : Sydney, Melbourne, Brisbane
      UTC-3  : São Paulo, Buenos Aires
      UTC-5  : New York, Toronto, Lima
      UTC-6  : Chicago, Mexico City
      UTC-8  : Los Angeles, Vancouver, Seattle
      UTC-10 : Honolulu

    Args:
        time_str  : Local time in HH:MM (24-hour format), e.g. "14:00"
        from_city : Source city name (case-insensitive)
        to_city   : Destination city name (case-insensitive)

    Returns:
        The equivalent local time in the destination city, with a day offset note
        if the conversion crosses midnight.
    """
    offsets = {
        "london": 0, "dublin": 0, "accra": 0,
        "paris": 1, "berlin": 1, "amsterdam": 1, "rome": 1, "madrid": 1,
        "cairo": 2, "helsinki": 2, "athens": 2,
        "moscow": 3, "riyadh": 3, "nairobi": 3,
        "dubai": 4,
        "mumbai": 5.5, "new delhi": 5.5, "kolkata": 5.5,
        "singapore": 8, "hong kong": 8, "beijing": 8, "shanghai": 8, "perth": 8,
        "tokyo": 9, "seoul": 9,
        "sydney": 10, "melbourne": 10, "brisbane": 10,
        "auckland": 12,
        "sao paulo": -3, "são paulo": -3, "buenos aires": -3,
        "new york": -5, "toronto": -5, "lima": -5,
        "chicago": -6, "mexico city": -6,
        "los angeles": -8, "vancouver": -8, "seattle": -8,
        "honolulu": -10,
    }

    f = from_city.lower().strip()
    t = to_city.lower().strip()

    if f not in offsets:
        return f"Unknown city '{from_city}'. See the tool docstring for supported cities."
    if t not in offsets:
        return f"Unknown city '{to_city}'. See the tool docstring for supported cities."

    try:
        h, m = map(int, time_str.strip().split(":"))
    except ValueError:
        return f"Invalid time format '{time_str}'. Use HH:MM (24-hour), e.g. '14:00'."

    total_min  = h * 60 + m
    utc_min    = total_min - int(offsets[f] * 60)
    dest_min   = utc_min   + int(offsets[t] * 60)

    day_note = ""
    if dest_min >= 1440:
        dest_min -= 1440
        day_note = " (next day)"
    elif dest_min < 0:
        dest_min += 1440
        day_note = " (previous day)"

    dh, dm = dest_min // 60, dest_min % 60
    to_off  = offsets[t]

    return (
        f"{dh:02d}:{dm:02d} local time in {to_city.title()}"
        f" (UTC{to_off:+.4g}){day_note}"
    )


print("timezone_convert('20:11', 'London', 'New York') →",
      timezone_convert.invoke({"time_str": "20:11", "from_city": "London", "to_city": "New York"}))
print("timezone_convert('14:00', 'London', 'Tokyo') →",
      timezone_convert.invoke({"time_str": "14:00", "from_city": "London", "to_city": "Tokyo"}))

## The Loop: Pseudocode vs. Reality

Before building the agent, let us map the abstract loop directly to the LangChain code that implements it.

```
PSEUDOCODE                              LANGCHAIN EQUIVALENT
──────────────────────────────────────  ────────────────────────────────────────────────
while not done:                         create_agent runs this graph internally

    response = call_llm(messages)       agent node: llm.invoke(messages + tool schema)

    if response has tool_calls:         graph routes to "tools" node if tool_calls exist

        results = execute_tools(...)        each tool is looked up by name and called

        messages.append(results)            ToolMessage results appended to state

    else:                               else:
        done = True                         graph routes to END
        return response                     final AIMessage content is the answer
```

`create_agent` returns a compiled `StateGraph`. It has two nodes: `agent` (the LLM reasoning step) and `tools` (tool execution). It routes between them until the LLM returns a message with no tool calls. The graph stops when:
- The LLM returns a response with **no tool calls** (task complete)
- The graph's `recursion_limit` is reached (pass `config={"recursion_limit": N}` to `.stream()`)
- An unrecoverable error is raised

Every time through the loop, the tool results accumulate in the graph's `messages` state. We stream this state so you can watch each iteration as it happens.

In [ ]:
# This cell shows the while loop running manually — without create_agent —
# to make the iteration structure explicit before we hand it to LangChain.

from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

tools_map = {
    "calculate":        calculate,
    "convert_units":    convert_units,
    "timezone_convert": timezone_convert,
}

# Bind tools so the LLM knows what it can call
llm_with_tools = llm.bind_tools(list(tools_map.values()))

# A minimal manual loop — the same logic create_agent encapsulates
def run_loop_manually(question: str, max_iter: int = 8):
    messages = [HumanMessage(content=question)]

    for iteration in range(1, max_iter + 1):
        print(f"\n{'─' * 50}")
        print(f"Iteration {iteration}  [REASON]")

        response = llm_with_tools.invoke(messages)   # REASON stage
        messages.append(response)

        if not response.tool_calls:                  # No tools → DONE
            print(f"  No tool calls → loop exits")
            print(f"\nFinal answer: {response.content}")
            return response.content

        for call in response.tool_calls:             # ACT stage
            print(f"  [ACT]     → {call['name']}({call['args']})")
            tool_fn = tools_map[call["name"]]
            result  = tool_fn.invoke(call["args"])
            print(f"  [OBSERVE] ← {result}")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

    return "Max iterations reached."


# Run the loop on a simple warm-up question
_ = run_loop_manually("How many minutes are in 6.189 hours?")

## Building the Agent

Now we construct the agent using LangChain's `create_agent`: the current API (LangChain 1.x) for building a tool-calling agent loop. Under the hood it compiles a `StateGraph` — the same graph-based runtime that LangGraph uses — so streaming and state management work identically.

`create_agent` takes a model, a tool list, and a system prompt, and returns a compiled graph. We call `.stream()` on it in the next cell to watch each iteration execute as it happens.

The `system_prompt` parameter prepends a `SystemMessage` before every LLM call. It tells the agent what tools it has and how to approach multi-step reasoning.

In [ ]:
from langchain.agents import create_agent

# ── Tool list ─────────────────────────────────────────────────────────────────
tools = [calculate, convert_units, timezone_convert]

# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a precise reasoning assistant with access to three tools:
- calculate: for arithmetic and mathematical expressions
- convert_units: for converting between distance, speed, and time units
- timezone_convert: for converting times between cities

When solving a problem:
1. Break it into steps and solve each step with a tool call.
2. Show your reasoning between steps: explain what each result means and what you will do next.
3. Do not guess or approximate — use tools for all numeric calculations.
4. When you have a complete answer, state it clearly in plain language."""

# ── Agent graph ───────────────────────────────────────────────────────────────
# create_agent returns a compiled StateGraph. The loop runs until the LLM
# returns a response with no tool calls, or the recursion_limit is reached.
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

print(f"Agent ready. Tools: {[t.name for t in tools]}")
print("Max iterations: controlled by recursion_limit in config (default 25).")

## Run the Agent

The cell below runs the agent on a multi-step reasoning question that requires all three tools.

Watch the output stream live. You will see the loop in action:

| Output marker | Loop stage |
|---------------|------------|
| `[ACT] → tool_name(args)` | **ACT** — the LLM has decided on a tool call |
| `[OBSERVE] ← result` | **OBSERVE** — the tool result feeds back into the graph state |
| `Answer: ...` | Loop exits: the LLM returned a final response with no further tool calls |

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage

QUESTION = (
    "A flight from London to New York JFK covers 5,570 km. "
    "The aircraft cruises at 900 km/h. "
    "The flight departs London at 14:00 local time. "
    "How long is the flight in hours and minutes, "
    "and what local time does it arrive in New York?"
)

print(f"Question: {QUESTION}\n")
print("=" * 60)

# ── Stream the agent ──────────────────────────────────────────────────────────
# stream_mode="values" yields the full messages list after every graph node.
# chunk["messages"][-1] is always the most recent event.

tool_calls_made = []   # List of dicts: {tool, input, result}
final_answer    = ""
pending_calls   = {}   # Maps tool_call_id → {tool, input} until result arrives

for chunk in agent.stream(
    {"messages": [("human", QUESTION)]},
    stream_mode="values",
):
    last_msg = chunk["messages"][-1]

    if isinstance(last_msg, AIMessage) and last_msg.tool_calls:
        # ── REASON + ACT stage: LLM has decided on tool calls ─────────────────
        for call in last_msg.tool_calls:
            print(f"\nIteration {len(tool_calls_made) + len(pending_calls) + 1}")
            print(f"  [ACT]     → {call['name']}({call['args']})")
            pending_calls[call["id"]] = {"tool": call["name"], "input": call["args"]}

    elif isinstance(last_msg, ToolMessage):
        # ── OBSERVE stage: tool result has come back ───────────────────────────
        call_id = last_msg.tool_call_id
        if call_id in pending_calls:
            entry = pending_calls.pop(call_id)
            result_str = str(last_msg.content)
            print(f"  [OBSERVE] ← {result_str}")
            tool_calls_made.append({
                "tool":   entry["tool"],
                "input":  entry["input"],
                "result": result_str,
            })

    elif isinstance(last_msg, AIMessage) and last_msg.content:
        # ── Final answer: LLM returned content with no tool calls ──────────────
        final_answer = last_msg.content

print("\n" + "=" * 60)
print(f"\nAnswer: {final_answer}")

## Inspect the Loop

The verbose output above shows the loop as it ran. The cell below presents the collected tool calls as a clean iteration-by-iteration trace — every tool the agent called, what it passed in, and what came back.

This is how you understand *why* the agent took the path it did.

In [ ]:
print(f"Loop completed in {len(tool_calls_made)} iteration(s)\n")
print("=" * 60)

for i, step in enumerate(tool_calls_made, 1):
    print(f"\nIteration {i}")
    print(f"  Stage    : ACT")
    print(f"  Tool     : {step['tool']}")
    print(f"  Input    : {step['input']}")
    print(f"  Stage    : OBSERVE")
    obs = str(step["result"])
    print(f"  Result   : {obs[:200]}{'...' if len(obs) > 200 else ''}")

print(f"\n{'=' * 60}")
print(f"Total tool calls: {len(tool_calls_made)}")

## Store the Trace in Oracle AI Database 26ai

Now we write the agent's execution trace to Oracle. Each iteration of the loop becomes one row in the `agent_traces` table — a permanent, queryable record of how the agent solved the problem.

Every run gets a unique `run_id` (a UUID), so you can accumulate many runs and query across all of them.

In [ ]:
import uuid
import json

run_id = str(uuid.uuid4())

INSERT_SQL = """
INSERT INTO agent_traces (run_id, iteration, tool_called, tool_input, tool_result, tokens_used)
VALUES (:run_id, :iteration, :tool_called, :tool_input, :tool_result, :tokens_used)
"""

with pool.acquire() as conn:
    with conn.cursor() as cur:
        for i, step in enumerate(tool_calls_made, 1):
            cur.execute(INSERT_SQL, {
                "run_id":      run_id,
                "iteration":   i,
                "tool_called": step["tool"],
                "tool_input":  json.dumps(step["input"]),
                "tool_result": step["result"],
                "tokens_used": None,
            })
    conn.commit()

print(f"Run {run_id} stored.")
print(f"{len(tool_calls_made)} iteration row(s) written to agent_traces.\n")

# Read it back immediately to confirm
with pool.acquire() as conn:
    with conn.cursor() as cur:
        cur.execute(
            "SELECT iteration, tool_called, tool_input FROM agent_traces "
            "WHERE run_id = :rid ORDER BY iteration",
            {"rid": run_id}
        )
        rows = cur.fetchall()

print(f"{'Iter':>4}  {'Tool':<20}  Input")
print("-" * 60)
for row in rows:
    inp = str(row[2])[:50] if row[2] else ""
    print(f"{row[0]:>4}  {row[1]:<20}  {inp}")

## Query Your Agent History

Run the agent a few more times with different questions, then use these SQL queries to explore patterns across all of your runs.

In [ ]:
# ── Second run: a different question to build up query history ────────────────
QUESTION2 = (
    "A marathon is 42.195 km. The world record is 2 hours, 0 minutes, and 35 seconds. "
    "What was the average speed in km/h and in mph?"
)

tool_calls_made2 = []
pending_calls2   = {}
final_answer2    = ""

for chunk in agent.stream(
    {"messages": [("human", QUESTION2)]},
    stream_mode="values",
):
    last_msg = chunk["messages"][-1]

    if isinstance(last_msg, AIMessage) and last_msg.tool_calls:
        for call in last_msg.tool_calls:
            pending_calls2[call["id"]] = {"tool": call["name"], "input": call["args"]}

    elif isinstance(last_msg, ToolMessage):
        call_id = last_msg.tool_call_id
        if call_id in pending_calls2:
            entry = pending_calls2.pop(call_id)
            tool_calls_made2.append({
                "tool":   entry["tool"],
                "input":  entry["input"],
                "result": str(last_msg.content),
            })

    elif isinstance(last_msg, AIMessage) and last_msg.content:
        final_answer2 = last_msg.content

run_id2 = str(uuid.uuid4())

with pool.acquire() as conn:
    with conn.cursor() as cur:
        for i, step in enumerate(tool_calls_made2, 1):
            cur.execute(INSERT_SQL, {
                "run_id":      run_id2,
                "iteration":   i,
                "tool_called": step["tool"],
                "tool_input":  json.dumps(step["input"]),
                "tool_result": step["result"],
                "tokens_used": None,
            })
    conn.commit()

print(f"Second run stored ({len(tool_calls_made2)} iterations). Running SQL queries...\n")

with pool.acquire() as conn:
    with conn.cursor() as cur:

        # 1. How many iterations did each run take?
        print("── Iterations per run ──────────────────────────")
        cur.execute(
            "SELECT run_id, COUNT(*) AS iterations "
            "FROM agent_traces GROUP BY run_id ORDER BY MIN(ts)"
        )
        for row in cur.fetchall():
            print(f"  {row[0][:8]}...  {row[1]} iteration(s)")

        # 2. Which tool was called most often?
        print("\n── Most-used tools (all runs) ──────────────────")
        cur.execute(
            "SELECT tool_called, COUNT(*) AS calls "
            "FROM agent_traces "
            "GROUP BY tool_called ORDER BY calls DESC"
        )
        for row in cur.fetchall():
            print(f"  {row[0]:<25}  {row[1]} call(s)")

        # 3. Full chronological trace across all runs
        print("\n── Full trace (all runs, chronological) ────────")
        cur.execute(
            "SELECT run_id, iteration, tool_called, ts "
            "FROM agent_traces ORDER BY ts"
        )
        for row in cur.fetchall():
            print(f"  {row[0][:8]}... | iter {row[1]} | {row[2]:<22} | {row[3]}")

## What Controls the Loop?

The agent loop is not infinite — it has stopping conditions. Understanding them is essential for building agents that behave predictably in production.

### Stopping conditions

| Condition | What happens | How to configure |
|-----------|-------------|------------------|
| **Goal achieved** | The LLM returns a response with no tool calls: it believes the task is complete | Achieved automatically; the quality of the system prompt affects how reliably this happens |
| **Recursion limit** | The graph has cycled `recursion_limit` times without finishing | Pass `config={"recursion_limit": N}` to `agent.stream()` — default is 25 |
| **Error escalation** | A tool raises an unrecoverable exception | Wrap tool logic in try/except and return an error string; the LLM reads it and can self-correct |
| **Token budget** | The accumulated message history fills the context window | Use a token-counting tool the agent checks before expensive calls |

### The danger of runaway loops

Without a `recursion_limit`, a confused agent can loop indefinitely — calling the same tools with the same arguments, never making progress. Always set a limit, and consider adding a token counter as a tool the agent checks before expensive operations.

### From demo to production

This notebook runs the agent loop synchronously in a notebook cell. In production you would typically:
- Run the loop **asynchronously** (`agent.astream()` / `agent.ainvoke()`)
- **Checkpoint** agent state between iterations (pass a `checkpointer=` to `create_agent`)
- **Monitor** each iteration with structured logging (OCI Logging, OpenTelemetry)
- Add **human-in-the-loop** approval for high-impact actions (`interrupt_before=["tools"]`)

Each of these patterns is documented in the Oracle AI Developer Hub notebooks.

## The Cost of the Loop

Stopping conditions are not just an academic concern — they are a cost control mechanism.

Agents consume **roughly 4x more tokens** than a standard single-turn chat interaction. Multi-agent systems, where agents spawn sub-agents, can reach **15x more tokens** than a basic chat call. For an agent running dozens of iterations across hundreds of daily tasks, this compounds quickly.

The consequences of getting this wrong can be swift. One production system called a broken tool 400 times in five minutes — accumulating thousands of tokens of tool results that went nowhere — before anyone noticed the loop was stuck. Stopping conditions prevented nothing, because none had been set.

The practical takeaway: **cost-per-task matters more than cost-per-token** when planning an agent deployment. Instrument your loop from day one. Log every iteration, track token consumption per run, and set `recursion_limit` before you go anywhere near production.

## Learn More

### Companion blog post
[**The AI agent loop: the architecture powering autonomous AI**](BLOG_URL) — Oracle Developer Blog  
The conceptual deep-dive behind this notebook: how OpenAI, Anthropic, Google, Microsoft, Meta, and LangChain each implement and extend the same core loop, and what the industry has converged on.

### Oracle AI Developer Hub
More notebooks on agent memory, RAG retrieval, reasoning strategies, and multi-agent architectures:  
[github.com/oracle-devrel/oracle-ai-developer-hub](https://github.com/oracle-devrel/oracle-ai-developer-hub)

### Reference documentation
- [Oracle 26ai Free container image](https://container-registry.oracle.com/ords/ocr/ba/database/free)
- [python-oracledb documentation](https://python-oracledb.readthedocs.io/)
- [Google AI Studio — get an API key](https://aistudio.google.com/apikey)
- [LangChain Google Generative AI integration](https://python.langchain.com/docs/integrations/chat/google_generative_ai/)
- [LangChain `create_agent` reference](https://docs.langchain.com/oss/python/langchain/agents)

### Foundational reading
- [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629) — the research paper that formalised the Thought → Action → Observation loop
- [Anthropic: Building Effective Agents](https://www.anthropic.com/research/building-effective-agents) — the most-cited industry guide on agent architecture